# Aggressive Fraud Risk Pipeline (IEEE-CIS)
### Goal: maximize PR-AUC / ROC-AUC with stronger features, tuning, and ensemble

**Reality check (read this):**
- **ROC-AUC > 0.80** is already realistic (you had ~0.88).
- **PR-AUC > 0.80** on IEEE-CIS with ~3.5% fraud is **hard** — Kaggle-level effort, full data, heavy features. This notebook pushes toward that; it does **not** guarantee 0.80 PR-AUC.
- Primary metric for imbalance: **PR-AUC**. We also report ROC-AUC, MCC, FC@budget, cost policy.

**Upgrades vs previous notebook:**
1. Composite entity key for behavioral history
2. Richer leakage-safe behavioral + time features
3. Frequency / smoothed target-style counts (train-only maps)
4. Tuned LightGBM + XGBoost + optional CatBoost
5. Soft-vote ensemble
6. Episodic parquet offload + float32
7. Cost-aware APPROVE/REVIEW/BLOCK + review budgets still included


## 0 — Setup


In [ ]:
import os, gc, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, precision_recall_curve,
)
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("NEED lightgbm")

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
    print("catboost optional — pip install catboost for extra boost")

try:
    from imblearn.over_sampling import SMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def rss_mb():
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024.0
    except Exception:
        return -1.0
    return -1.0

print("LGB", HAS_LGB, "XGB", HAS_XGB, "CAT", HAS_CAT, f"RSS={rss_mb():.0f}")


In [ ]:
CONFIG = {
    "ieee_transaction_path": "/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv",
    "ieee_identity_path": "/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv",

    # Use 300_000 or None on high RAM for better scores
    "SAMPLE_N_IEEE": 300_000,

    "train_frac": 0.60,
    "val_frac": 0.20,
    "test_frac": 0.20,

    "drop_high_missing": True,
    "high_missing_threshold": 0.92,

    "velocity_windows": [3600 * h for h in [1, 6, 12, 24, 72, 168]],

    "output_dir": "/kaggle/working/outputs_sota",
    "feat_parquet": "/kaggle/working/outputs_sota/ieee_feat_sota.parquet",
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if "path" not in k}, indent=2))


## 1 — Load + composite entity + downcast


In [ ]:
def downcast_df(df):
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

def load_ieee(config):
    t0 = time.time()
    nrows = config["SAMPLE_N_IEEE"]
    print(f"Loading transaction nrows={nrows}...", flush=True)
    df = pd.read_csv(config["ieee_transaction_path"], nrows=nrows)
    df = downcast_df(df)

    ident_path = Path(config["ieee_identity_path"])
    if ident_path.exists():
        ids = set(df["TransactionID"].values)
        chunks = []
        for ch in pd.read_csv(ident_path, chunksize=250_000):
            ch = ch[ch["TransactionID"].isin(ids)]
            if len(ch):
                chunks.append(ch)
        if chunks:
            ident = downcast_df(pd.concat(chunks, ignore_index=True))
            df = df.merge(ident, on="TransactionID", how="left")
            del ident, chunks
            gc.collect()

    if config.get("drop_high_missing"):
        thr = config["high_missing_threshold"]
        miss = df.isna().mean()
        protect = {
            "isFraud", "TransactionID", "TransactionDT", "TransactionAmt",
            "card1", "addr1", "P_emaildomain", "ProductCD",
        }
        drop = [c for c in miss.index if miss[c] > thr and c not in protect]
        print(f"Drop {len(drop)} high-missing cols", flush=True)
        df = df.drop(columns=drop)

    # Composite entity for behavioral history (stronger than card1 alone)
    df["entity_key"] = (
        df["card1"].astype(str).fillna("NA") + "|" +
        df.get("addr1", pd.Series(["NA"] * len(df))).astype(str).fillna("NA") + "|" +
        df.get("P_emaildomain", pd.Series(["NA"] * len(df))).astype(str).fillna("NA")
    )

    df = df.sort_values("TransactionDT").reset_index(drop=True)
    df = downcast_df(df)
    print(f"Loaded {df.shape} fraud={df['isFraud'].mean()*100:.3f}% in {time.time()-t0:.1f}s RSS={rss_mb():.0f}")
    return df

df = load_ieee(CONFIG)


## 2 — Leakage-safe behavioral + time features (vectorized)


In [ ]:
def add_behavioral_features(df, entity_col="entity_key", amount_col="TransactionAmt",
                            time_col="TransactionDT", windows=None, eps=1e-6):
    windows = windows or CONFIG["velocity_windows"]
    t0 = time.time()
    df = df.sort_values(time_col).reset_index(drop=True)
    g = df.groupby(entity_col, sort=False)

    df["beh_prev_tx_count"] = g.cumcount().astype(np.int32)

    shifted = g[amount_col].shift(1)
    past_cnt = df["beh_prev_tx_count"].astype(np.float32)
    past_sum = shifted.fillna(0).groupby(df[entity_col], sort=False).cumsum()
    df["beh_prev_amount_mean"] = np.where(past_cnt > 0, past_sum / past_cnt, np.nan).astype(np.float32)

    past_sum_sq = (shifted ** 2).fillna(0).groupby(df[entity_col], sort=False).cumsum()
    mean_sq = np.where(past_cnt > 0, past_sum_sq / past_cnt, np.nan)
    var = np.maximum(mean_sq - df["beh_prev_amount_mean"].astype(np.float64) ** 2, 0)
    df["beh_prev_amount_std"] = np.sqrt(var).astype(np.float32).fillna(0)

    # Robust median of past amounts via expanding median is slow; use shifted rolling approx per entity skip — keep mean-based
    df["beh_amount_zscore"] = ((df[amount_col] - df["beh_prev_amount_mean"]) / (df["beh_prev_amount_std"] + eps)).astype(np.float32)
    df["beh_amount_ratio"] = (df[amount_col] / (df["beh_prev_amount_mean"] + eps)).astype(np.float32)
    first = df["beh_prev_tx_count"] == 0
    df.loc[first, ["beh_amount_zscore", "beh_amount_ratio"]] = 0.0
    df["beh_amount_ratio"] = df["beh_amount_ratio"].replace([np.inf, -np.inf], np.nan)
    med = df["beh_amount_ratio"].median()
    df["beh_amount_ratio"] = df["beh_amount_ratio"].fillna(med if pd.notna(med) else 1.0)

    df["beh_time_since_prev"] = (df[time_col] - g[time_col].shift(1)).astype(np.float32)
    mx = df["beh_time_since_prev"].max()
    df["beh_time_since_prev"] = df["beh_time_since_prev"].fillna(mx if pd.notna(mx) else 0)

    # Log1p amount + time-of-day from TransactionDT (seconds)
    df["beh_log_amount"] = np.log1p(df[amount_col].astype(np.float32))
    tod = (df[time_col] % 86400).astype(np.float32)
    df["beh_hour_sin"] = np.sin(2 * np.pi * tod / 86400).astype(np.float32)
    df["beh_hour_cos"] = np.cos(2 * np.pi * tod / 86400).astype(np.float32)
    df["beh_dow_sin"] = np.sin(2 * np.pi * (df[time_col] % (86400 * 7)) / (86400 * 7)).astype(np.float32)
    df["beh_dow_cos"] = np.cos(2 * np.pi * (df[time_col] % (86400 * 7)) / (86400 * 7)).astype(np.float32)

    # Velocity two-pointer
    print("  velocity windows...", flush=True)
    entities = df[entity_col].fillna("__NA__").values
    times = df[time_col].values
    order = np.argsort(entities, kind="stable")
    ent_vals, t_vals = entities[order], times[order]
    for w in windows:
        counts = np.zeros(len(df), dtype=np.int32)
        n, i = len(df), 0
        while i < n:
            j = i
            while j < n and ent_vals[j] == ent_vals[i]:
                j += 1
            sub = t_vals[i:j]
            lo = 0
            for k in range(len(sub)):
                cutoff = sub[k] - w
                while lo < k and sub[lo] < cutoff:
                    lo += 1
                counts[i + k] = k - lo
            i = j
        out = np.empty(len(df), dtype=np.int32)
        out[order] = counts
        df[f"beh_velocity_{w}s"] = out

    # Past fraud rate per entity (strictly past labels only) — powerful if stable
    # Uses shift so current label never leaks
    if "isFraud" in df.columns:
        shifted_y = g["isFraud"].shift(1)
        past_fraud_sum = shifted_y.fillna(0).groupby(df[entity_col], sort=False).cumsum()
        df["beh_prev_fraud_rate"] = np.where(past_cnt > 0, past_fraud_sum / past_cnt, 0).astype(np.float32)
        df["beh_prev_fraud_count"] = past_fraud_sum.astype(np.float32)

    print(f"  behavioral done {time.time()-t0:.1f}s shape={df.shape} RSS={rss_mb():.0f}")
    return df

df = add_behavioral_features(df)
beh_cols = [c for c in df.columns if c.startswith("beh_")]
print("beh cols", len(beh_cols), beh_cols[:12], "...")


In [ ]:
# Save features and free
path = CONFIG["feat_parquet"]
df.to_parquet(path, index=False)
print(f"Saved {path} rows={len(df):,}")
del df
gc.collect()
print(f"RSS={rss_mb():.0f} — restart kernel optional, then re-run setup + load parquet")


## 3 — Split + preprocess (freq encode + float32)


In [ ]:
df = pd.read_parquet(CONFIG["feat_parquet"])
print(df.shape, f"RSS={rss_mb():.0f}")

def temporal_split(df, time_col="TransactionDT"):
    df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df)
    a, b = int(n * CONFIG["train_frac"]), int(n * (CONFIG["train_frac"] + CONFIG["val_frac"]))
    tr, va, te = df.iloc[:a].copy(), df.iloc[a:b].copy(), df.iloc[b:].copy()
    for name, d in [("train", tr), ("val", va), ("test", te)]:
        print(f"  {name}: {len(d):,} fraud={d['isFraud'].mean()*100:.3f}%")
    return tr, va, te

train_df, val_df, test_df = temporal_split(df)
del df
gc.collect()


In [ ]:
def build_feature_list(df):
    core = [c for c in [
        "TransactionAmt", "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
        "addr1", "addr2", "dist1", "dist2", "P_emaildomain", "R_emaildomain",
        "DeviceType", "DeviceInfo",
    ] if c in df.columns]
    groups = []
    for prefix in ("C", "D", "V", "M", "id_"):
        groups += [c for c in df.columns if c.startswith(prefix) and c not in core]
    beh = [c for c in df.columns if c.startswith("beh_")]
    # drop ultra-id
    ban = {"TransactionID", "isFraud", "TransactionDT", "entity_key"}
    feats = [c for c in core + groups + beh if c not in ban and c in df.columns]
    # de-dupe
    seen, out = set(), []
    for c in feats:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

FEATURE_COLS = build_feature_list(train_df)
print(f"n_features raw={len(FEATURE_COLS)}")

class Preprocessor:
    def __init__(self, cols, miss_thr=0.5):
        self.cols = cols
        self.miss_thr = miss_thr
        self.cat_cols, self.num_cols, self.flag_cols = [], [], []
        self.freq, self.med = {}, {}

    def fit(self, df):
        self.cols = [c for c in self.cols if c in df.columns]
        self.cat_cols = [c for c in self.cols if df[c].dtype == object or str(df[c].dtype) == "category"]
        self.num_cols = [c for c in self.cols if c not in self.cat_cols]
        miss = df[self.cols].isna().mean()
        self.flag_cols = miss[miss > self.miss_thr].index.tolist()
        for c in self.cat_cols:
            self.freq[c] = df[c].astype(str).value_counts(normalize=True).to_dict()
        for c in self.num_cols:
            self.med[c] = float(df[c].median()) if df[c].notna().any() else 0.0
        return self

    def transform(self, df):
        n = len(df)
        data = {}
        for c in self.flag_cols:
            data[f"{c}_miss"] = df[c].isna().astype(np.float32).values if c in df.columns else np.zeros(n, np.float32)
        for c in self.cat_cols:
            m = self.freq.get(c, {})
            data[c] = df[c].astype(str).map(m).fillna(0).astype(np.float32).values if c in df.columns else np.zeros(n, np.float32)
        for c in self.num_cols:
            med = self.med.get(c, 0.0)
            data[c] = df[c].fillna(med).astype(np.float32).values if c in df.columns else np.full(n, med, np.float32)
        return pd.DataFrame(data, index=df.index)

prep = Preprocessor(FEATURE_COLS).fit(train_df)
X_train = prep.transform(train_df)
X_val = prep.transform(val_df)
X_test = prep.transform(test_df)
y_train = train_df["isFraud"].values.astype(np.int8)
y_val = val_df["isFraud"].values.astype(np.int8)
y_test = test_df["isFraud"].values.astype(np.int8)
print(X_train.shape, X_val.shape, X_test.shape, f"RSS={rss_mb():.0f}")

# optional: free raw splits column-heavy frames if needed later keep y only


## 4 — Tuned models + ensemble (maximize VAL PR-AUC)


In [ ]:
def evaluate(y_true, proba, label=""):
    pred = (proba >= 0.5).astype(int)
    return {
        "label": label,
        "PR_AUC": float(average_precision_score(y_true, proba)),
        "ROC_AUC": float(roc_auc_score(y_true, proba)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "MCC": float(matthews_corrcoef(y_true, pred)),
    }

pos_w = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print("pos_weight", round(pos_w, 2))

results = []
oof_val = {}
oof_test = {}

# ---- LightGBM grid (small but effective) ----
if HAS_LGB:
    best_lgb, best_score, best_params = None, -1, None
    grid = [
        dict(num_leaves=63, learning_rate=0.03, min_child_samples=40, reg_lambda=1.0),
        dict(num_leaves=127, learning_rate=0.03, min_child_samples=30, reg_lambda=2.0),
        dict(num_leaves=63, learning_rate=0.05, min_child_samples=20, reg_lambda=1.0),
        dict(num_leaves=255, learning_rate=0.02, min_child_samples=50, reg_lambda=3.0),
    ]
    for i, p in enumerate(grid):
        print(f"LGB trial {i+1}/{len(grid)} {p}", flush=True)
        m = lgb.LGBMClassifier(
            n_estimators=800, subsample=0.85, colsample_bytree=0.7,
            scale_pos_weight=pos_w, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
            **p,
        )
        m.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="average_precision",
            callbacks=[lgb.early_stopping(60, verbose=False)],
        )
        pv = m.predict_proba(X_val)[:, 1]
        sc = average_precision_score(y_val, pv)
        print(f"  VAL PR-AUC={sc:.4f}")
        if sc > best_score:
            best_score, best_lgb, best_params = sc, m, p
    pv = best_lgb.predict_proba(X_val)[:, 1]
    pt = best_lgb.predict_proba(X_test)[:, 1]
    oof_val["lgb"], oof_test["lgb"] = pv, pt
    r = evaluate(y_test, pt, "LGBM_TEST")
    results.append(r)
    print("BEST LGB", best_params, "TEST", r)
    gc.collect()

# ---- XGBoost ----
if HAS_XGB:
    print("Fitting XGBoost...", flush=True)
    mx = xgb.XGBClassifier(
        n_estimators=600, max_depth=7, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.7, min_child_weight=5,
        reg_lambda=2.0, scale_pos_weight=pos_w,
        eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1,
        tree_method="hist", early_stopping_rounds=50,
    )
    mx.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pv = mx.predict_proba(X_val)[:, 1]
    pt = mx.predict_proba(X_test)[:, 1]
    oof_val["xgb"], oof_test["xgb"] = pv, pt
    r = evaluate(y_test, pt, "XGB_TEST")
    results.append(r)
    print("XGB TEST", r)
    gc.collect()

# ---- CatBoost ----
if HAS_CAT:
    print("Fitting CatBoost...", flush=True)
    mc = CatBoostClassifier(
        iterations=800, depth=8, learning_rate=0.03,
        l2_leaf_reg=4.0, random_seed=RANDOM_STATE, verbose=False,
        auto_class_weights="Balanced", eval_metric="PRAUC",
        early_stopping_rounds=50,
    )
    mc.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    pv = mc.predict_proba(X_val)[:, 1]
    pt = mc.predict_proba(X_test)[:, 1]
    oof_val["cat"], oof_test["cat"] = pv, pt
    r = evaluate(y_test, pt, "CAT_TEST")
    results.append(r)
    print("CAT TEST", r)
    gc.collect()


In [ ]:
# Soft-vote ensemble + VAL-weighted blend
names = list(oof_val.keys())
assert names, "No models trained"

# Optimize blend weights on VAL PR-AUC (simple grid for 2-3 models)
from itertools import product

def blend_scores(weights, probs):
    w = np.array(weights, dtype=float)
    w = w / w.sum()
    return sum(w[i] * probs[i] for i in range(len(probs)))

val_probs = [oof_val[n] for n in names]
test_probs = [oof_test[n] for n in names]

best_w, best_sc = None, -1
if len(names) == 1:
    best_w = [1.0]
else:
    grid = [0.0, 0.25, 0.5, 0.75, 1.0]
    for ws in product(grid, repeat=len(names)):
        if sum(ws) == 0:
            continue
        sc = average_precision_score(y_val, blend_scores(ws, val_probs))
        if sc > best_sc:
            best_sc, best_w = sc, ws

w = np.array(best_w, dtype=float)
w = w / w.sum()
print("Blend weights", dict(zip(names, w.round(3))), "VAL PR-AUC", round(best_sc if best_sc > 0 else average_precision_score(y_val, val_probs[0]), 4))

proba_val = blend_scores(w, val_probs)
proba_test = blend_scores(w, test_probs)
ens = evaluate(y_test, proba_test, "ENSEMBLE_TEST")
results.append(ens)
print("ENSEMBLE TEST", ens)

df_res = pd.DataFrame(results).sort_values("PR_AUC", ascending=False)
print(df_res)
df_res.to_csv(f"{CONFIG['output_dir']}/model_scores.csv", index=False)

print("\n=== TARGET CHECK ===")
print(f"ROC_AUC={ens['ROC_AUC']:.4f}  (>=0.80? {ens['ROC_AUC']>=0.80})")
print(f"PR_AUC ={ens['PR_AUC']:.4f}  (>=0.80? {ens['PR_AUC']>=0.80})  # stretch goal")


## 5 — Optional SMOTE on best single model (compare)


In [ ]:
if HAS_IMBLEARN and HAS_LGB:
    print("SMOTE on TRAIN only + retune short LGB...", flush=True)
    sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    Xtr_s, ytr_s = sm.fit_resample(X_train, y_train)
    pos_ws = (ytr_s == 0).sum() / max((ytr_s == 1).sum(), 1)
    m = lgb.LGBMClassifier(
        n_estimators=600, num_leaves=63, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.7, scale_pos_weight=1.0,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )
    m.fit(Xtr_s, ytr_s, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    pt = m.predict_proba(X_test)[:, 1]
    r = evaluate(y_test, pt, "LGBM_SMOTE_TEST")
    results.append(r)
    print(r)
    del Xtr_s, ytr_s
    gc.collect()
else:
    print("Skip SMOTE")


## 6 — Risk-to-action + review budgets (same business layer)


In [ ]:
def expected_cost(y_true, proba, tau1, tau2, c_fn, c_fp, c_review):
    action = np.where(proba < tau1, "APPROVE", np.where(proba < tau2, "REVIEW", "BLOCK"))
    cost = np.zeros(len(y_true))
    cost[(action == "APPROVE") & (y_true == 1)] = c_fn
    cost[(action == "BLOCK") & (y_true == 0)] = c_fp
    cost[action == "REVIEW"] = c_review
    return {
        "tau1": float(tau1), "tau2": float(tau2), "total_cost": float(cost.sum()),
        "fraud_capture": float(y_true[action != "APPROVE"].sum() / max(y_true.sum(), 1)),
        "review_rate": float((action == "REVIEW").mean()),
        "false_decline_rate": float(((action == "BLOCK") & (y_true == 0)).sum() / max((y_true == 0).sum(), 1)),
    }

def optimize_thresholds(y_true, proba, c_fn, c_fp, c_review, grid=41):
    taus = np.linspace(0.01, 0.99, grid)
    best = None
    for t1 in taus:
        for t2 in taus:
            if t2 <= t1:
                continue
            r = expected_cost(y_true, proba, t1, t2, c_fn, c_fp, c_review)
            if best is None or r["total_cost"] < best["total_cost"]:
                best = r
    return best

for name, cfn in [("S1_10x", 10), ("S2_25x", 25), ("S3_50x", 50)]:
    opt = optimize_thresholds(y_test, proba_test, cfn, 1.0, 0.2)
    conv = expected_cost(y_test, proba_test, 0.5, 0.5 + 1e-9, cfn, 1.0, 0.2)
    print(f"{name} OPT cost={opt['total_cost']:.0f} capture={opt['fraud_capture']:.3f} review={opt['review_rate']:.3f} | "
          f"CONV cost={conv['total_cost']:.0f} capture={conv['fraud_capture']:.3f}")

order = np.argsort(-proba_test)
y_sorted = y_test[order]
total_fraud = y_test.sum()
for b in (0.01, 0.05, 0.10):
    k = max(int(np.ceil(len(y_test) * b)), 1)
    print(f"FC@{int(b*100)}% = {y_sorted[:k].sum() / max(total_fraud, 1):.3f}")


## 7 — SHAP on best available tree model


In [ ]:
if HAS_SHAP and HAS_LGB and "best_lgb" in dir():
    sample = X_test.sample(n=min(1200, len(X_test)), random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(best_lgb)
    sv = explainer.shap_values(sample)
    if isinstance(sv, list):
        sv = sv[1]
    shap.summary_plot(sv, sample, plot_type="bar", show=True)
    shap.summary_plot(sv, sample, show=True)
else:
    print("SHAP skipped")


In [ ]:
summary = {
    "sample_n": CONFIG["SAMPLE_N_IEEE"],
    "n_features": int(X_train.shape[1]),
    "scores": results,
    "ensemble_test": ens,
    "note": "PR-AUC>=0.80 is a stretch on IEEE-CIS; ROC-AUC>=0.80 is the reliable bar.",
}
path = Path(CONFIG["output_dir"]) / "aggressive_results.json"
with open(path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print("Saved", path)


## How to push scores further

1. Set `SAMPLE_N_IEEE = None` on a machine with 30GB+ RAM and re-run.
2. Install CatBoost: `!pip install catboost` for a third ensemble member.
3. Do **not** use random split — keep temporal split (honest scores).
4. If PR-AUC still < 0.80: that is normal for this dataset under strict temporal validation; report ROC-AUC, MCC, FC@budget, and cost policy as the operational win.
